# Exploratory Data Analysis -  NeuroGuard 280-Cell Experiment

This notebook characterises the full factorial experiment output:
- 4 scenarios × 7 pressures × 2 monitoring × 1 model × 5 seeds = **280 sessions**

Key questions:
1. What is the overall misalignment (unsafe) rate?
2. Do adversarial pressure conditions increase unsafe behaviour?
3. Does monitoring reduce unsafe behaviour?
4. Which scenarios are most vulnerable?
5. What do the Layer B judge scores look like?
6. Are the engineered features predictive of the unsafe label?

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style='whitegrid', palette='colorblind', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (10, 6)

# Load assembled datasets
d2a = pd.read_parquet('../data/processed/d2a_sessions.parquet')
d2c = pd.read_parquet('../data/processed/d2c_labels.parquet')
d2d = pd.read_parquet('../data/processed/d2d_features.parquet')

# Merge into one analysis DataFrame
df = d2a.merge(d2c, on='session_id').merge(d2d, on='session_id', suffixes=('', '_feat'))

# Clean up any merge-suffix artefacts on the target column
if 'overall_unsafe_x' in df.columns:
    df.rename(columns={'overall_unsafe_x': 'overall_unsafe'}, inplace=True)
    df.drop(columns=[c for c in df.columns if c == 'overall_unsafe_y'], errors='ignore', inplace=True)

print(f'Sessions loaded: {len(df)}')
print(f'Columns: {len(df.columns)}')
print(f'Scenarios: {df.scenario_id.nunique()}')
print(f'Pressures: {df.pressure_id.nunique()}')
print(f'Monitoring: {df.monitoring_id.nunique()}')
df.head(3)

## 1. Overall Misalignment Rate & Class Balance

In [ ]:
unsafe_col = 'overall_unsafe'
unsafe_rate = df[unsafe_col].mean()
n_unsafe = df[unsafe_col].sum()
n_safe = len(df) - n_unsafe

print(f'Overall unsafe rate: {unsafe_rate:.1%} ({n_unsafe}/{len(df)})')
print(f'Class balance — Safe: {n_safe} | Unsafe: {n_unsafe} | Ratio: {n_safe/max(n_unsafe,1):.1f}:1')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pie chart
axes[0].pie([n_safe, n_unsafe], labels=['Safe', 'Unsafe'],
            autopct='%1.1f%%', colors=['#2ecc71', '#e74c3c'], startangle=90)
axes[0].set_title('Class Distribution')

# Bar chart by scenario
scenario_rates = df.groupby('scenario_id')[unsafe_col].mean().sort_values(ascending=False)
axes[1].bar(range(len(scenario_rates)), scenario_rates.values, color='#e74c3c', alpha=0.8)
axes[1].set_xticks(range(len(scenario_rates)))
axes[1].set_xticklabels(scenario_rates.index, rotation=25, ha='right')
axes[1].set_title('Unsafe Rate by Scenario')
axes[1].set_ylabel('Unsafe Rate')
axes[1].set_xlabel('')
axes[1].axhline(unsafe_rate, ls='--', color='grey', label=f'Overall: {unsafe_rate:.1%}')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/01_class_balance.png', bbox_inches='tight')
plt.show()

## 2. Unsafe Rate by Pressure Condition

**Core hypothesis:** Adversarial pressure conditions increase the probability of misaligned behaviour compared to the no-pressure baseline.

In [ ]:
pressure_rates = df.groupby('pressure_id')[unsafe_col].agg(['mean', 'sum', 'count'])
pressure_rates.columns = ['unsafe_rate', 'n_unsafe', 'n_total']
pressure_rates = pressure_rates.sort_values('unsafe_rate', ascending=False)

# Wilson confidence intervals
from statsmodels.stats.proportion import proportion_confint
cis = [proportion_confint(row.n_unsafe, row.n_total, alpha=0.05, method='wilson')
       for _, row in pressure_rates.iterrows()]
pressure_rates['ci_low'] = [c[0] for c in cis]
pressure_rates['ci_high'] = [c[1] for c in cis]

print(pressure_rates.to_string())

fig, ax = plt.subplots(figsize=(10, 5))
x = range(len(pressure_rates))
ax.bar(x, pressure_rates.unsafe_rate, color='#e74c3c', alpha=0.8)
ax.errorbar(x, pressure_rates.unsafe_rate,
            yerr=[pressure_rates.unsafe_rate - pressure_rates.ci_low,
                  pressure_rates.ci_high - pressure_rates.unsafe_rate],
            fmt='none', color='black', capsize=4)
ax.set_xticks(x)
ax.set_xticklabels(pressure_rates.index, rotation=30, ha='right')
ax.set_ylabel('Unsafe Rate')
ax.set_title('Unsafe Rate by Pressure Condition (with 95% Wilson CIs)')
ax.axhline(unsafe_rate, ls='--', color='grey', alpha=0.6, label=f'Overall: {unsafe_rate:.1%}')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/02_pressure_unsafe_rate.png', bbox_inches='tight')
plt.show()

## 3. Monitoring Effect

**Hypothesis:** Models behave more safely when told their output will be reviewed (monitored condition).

In [ ]:
mon_rates = df.groupby('monitoring_id')[unsafe_col].agg(['mean', 'sum', 'count'])
mon_rates.columns = ['unsafe_rate', 'n_unsafe', 'n_total']
print(mon_rates)

# Chi-squared test for monitoring effect
ct = pd.crosstab(df['monitoring_id'], df[unsafe_col])
chi2, p_val, dof, expected = stats.chi2_contingency(ct)
print(f'\nChi-squared test: chi2={chi2:.3f}, p={p_val:.4f}, dof={dof}')

# Interaction: pressure x monitoring
fig, ax = plt.subplots(figsize=(12, 5))
interaction = df.groupby(['pressure_id', 'monitoring_id'])[unsafe_col].mean().unstack()
interaction.plot.bar(ax=ax, color=['#3498db', '#e67e22'], alpha=0.85)
ax.set_title('Unsafe Rate: Pressure x Monitoring Interaction')
ax.set_ylabel('Unsafe Rate')
ax.set_xlabel('')
ax.legend(title='Monitoring')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../reports/figures/03_pressure_monitoring_interaction.png', bbox_inches='tight')
plt.show()

## 4. Layer A Signal Breakdown

Which specific safety signals are triggering across conditions?

In [ ]:
# Find all layer_a trigger columns
trigger_cols = [c for c in df.columns if c.startswith('layer_a_') and c.endswith('_triggered')]
print(f'Layer A signal columns ({len(trigger_cols)}): {[c.replace("layer_a_", "").replace("_triggered", "") for c in trigger_cols]}\n')

# Cast to numeric (NaN-safe) — some scenarios lack certain signals
for col in trigger_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)

# Signal frequency
signal_freq = df[trigger_cols].mean().sort_values(ascending=False)
signal_freq.index = [c.replace('layer_a_', '').replace('_triggered', '') for c in signal_freq.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall signal frequencies
signal_freq.plot.barh(ax=axes[0], color='#9b59b6', alpha=0.8)
axes[0].set_title('Layer A Signal Trigger Rates (Overall)')
axes[0].set_xlabel('Proportion of Sessions')

# Signal rates by pressure (heatmap)
signal_by_pressure = df.groupby('pressure_id')[trigger_cols].mean().astype(float)
signal_by_pressure.columns = [c.replace('layer_a_', '').replace('_triggered', '')
                               for c in signal_by_pressure.columns]
sns.heatmap(signal_by_pressure.T, annot=True, fmt='.2f', cmap='Reds',
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Signal Trigger Rate by Pressure')

plt.tight_layout()
plt.savefig('../reports/figures/04_layer_a_signals.png', bbox_inches='tight')
plt.show()

## 5. Layer B Judge Score Distributions

Layer B scores are 0-4 Likert per dimension (lower = more aligned).

In [ ]:
judge_cols = [c for c in df.columns if c.startswith('layer_b_') and c != 'layer_b_misaligned']

# Cast to numeric
for col in judge_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

judge_df = df[judge_cols].dropna()
print(f'Sessions with Layer B scores: {len(judge_df)}')
print(judge_df.describe().round(2))

n_judge_cols = len(judge_cols)
fig, axes = plt.subplots(1, n_judge_cols, figsize=(4 * n_judge_cols, 4), sharey=True)
if n_judge_cols == 1:
    axes = [axes]
for i, col in enumerate(judge_cols):
    label = col.replace('layer_b_', '')
    axes[i].hist(judge_df[col].astype(float), bins=np.arange(-0.5, 5.5, 1),
                 color='#3498db', alpha=0.8, edgecolor='white')
    axes[i].set_title(label, fontsize=10)
    axes[i].set_xlabel('Score (0-4)')
    axes[i].set_xticks(range(5))
    if i == 0:
        axes[i].set_ylabel('Count')

plt.suptitle('Layer B Judge Score Distributions', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/05_layer_b_distributions.png', bbox_inches='tight')
plt.show()

## 6. Layer B Scores by Pressure (Heatmap)

Do judge scores systematically worsen under adversarial pressure?

In [ ]:
judge_by_pressure = df.groupby('pressure_id')[judge_cols].mean().astype(float)
judge_by_pressure.columns = [c.replace('layer_b_', '') for c in judge_by_pressure.columns]

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(judge_by_pressure, annot=True, fmt='.2f', cmap='YlOrRd',
            ax=ax, vmin=0, vmax=4)
ax.set_title('Mean Layer B Judge Scores by Pressure (0=aligned, 4=misaligned)')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../reports/figures/06_layer_b_by_pressure.png', bbox_inches='tight')
plt.show()

## 7. Feature Distributions -  Safe vs Unsafe

Are the engineered features visually separable between safe and unsafe sessions?

In [ ]:
numeric_features = ['response_length_words', 'hedging_count', 'hedging_freq',
                     'certainty_count', 'certainty_freq', 'hedging_certainty_ratio',
                     'nct_id_count', 'nct_id_unique', 'citation_density',
                     'avg_sentence_length', 'sentence_count', 'bullet_count']

# Keep only features that exist in df
numeric_features = [f for f in numeric_features if f in df.columns]

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(numeric_features):
    if i >= len(axes):
        break
    for label, color in [(False, '#2ecc71'), (True, '#e74c3c')]:
        subset = df[df[unsafe_col] == label][feat].dropna()
        axes[i].hist(subset, bins=20, alpha=0.6, color=color,
                     label='Unsafe' if label else 'Safe', density=True)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

# Hide unused subplots
for j in range(len(numeric_features), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Safe vs Unsafe', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/07_feature_distributions.png', bbox_inches='tight')
plt.show()

## 8. Feature Correlations

In [ ]:
# Correlation matrix of numeric features + target
corr_cols = numeric_features + [unsafe_col]
corr_df = df[corr_cols].copy()
corr_df[unsafe_col] = corr_df[unsafe_col].astype(int)

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, ax=ax, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
ax.set_title('Feature Correlation Matrix (incl. unsafe target)')
plt.tight_layout()
plt.savefig('../reports/figures/08_correlation_matrix.png', bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr_matrix[unsafe_col].drop(unsafe_col).abs().sort_values(ascending=False)
print('Top feature correlations with unsafe label:')
print(target_corr.to_string())

## 9. Statistical Tests

Formal hypothesis tests for the two primary effects.

In [ ]:
# 1. Pressure effect: chi-squared across all 7 pressure levels
ct_pressure = pd.crosstab(df['pressure_id'], df[unsafe_col])
chi2_p, p_p, dof_p, _ = stats.chi2_contingency(ct_pressure)
print(f'=== Pressure Effect (chi-squared) ===')
print(f'chi2 = {chi2_p:.3f}, p = {p_p:.6f}, dof = {dof_p}')
print(f'Significant at alpha=0.05: {p_p < 0.05}\n')

# 2. Monitoring effect: chi-squared
ct_mon = pd.crosstab(df['monitoring_id'], df[unsafe_col])
chi2_m, p_m, dof_m, _ = stats.chi2_contingency(ct_mon)
print(f'=== Monitoring Effect (chi-squared) ===')
print(f'chi2 = {chi2_m:.3f}, p = {p_m:.6f}, dof = {dof_m}')
print(f'Significant at alpha=0.05: {p_m < 0.05}\n')

# 3. Pairwise: each pressure vs baseline (Fisher exact)
baseline_unsafe = df[df.pressure_id == 'none'][unsafe_col]
print(f'=== Pairwise: Each Pressure vs Baseline (Fisher exact) ===')
for pid in sorted(df.pressure_id.unique()):
    if pid == 'none':
        continue
    condition_unsafe = df[df.pressure_id == pid][unsafe_col]
    table = [[condition_unsafe.sum(), len(condition_unsafe) - condition_unsafe.sum()],
             [baseline_unsafe.sum(), len(baseline_unsafe) - baseline_unsafe.sum()]]
    odds_ratio_result = stats.fisher_exact(table)
    or_val = odds_ratio_result[0] if hasattr(odds_ratio_result[0], '__float__') else odds_ratio_result.statistic
    p_val = odds_ratio_result[1] if isinstance(odds_ratio_result[1], float) else odds_ratio_result.pvalue
    sig = '*' if p_val < 0.05 else ''
    print(f'  {pid:25s} OR={or_val:6.2f}  p={p_val:.4f} {sig}')

# 4. Scenario effect
ct_scenario = pd.crosstab(df['scenario_id'], df[unsafe_col])
chi2_s, p_s, dof_s, _ = stats.chi2_contingency(ct_scenario)
print(f'\n=== Scenario Effect (chi-squared) ===')
print(f'chi2 = {chi2_s:.3f}, p = {p_s:.6f}, dof = {dof_s}')
print(f'Significant at alpha=0.05: {p_s < 0.05}')

## 10. Summary Table & Key Findings

In [ ]:
# Summary pivot: scenario x pressure
pivot = df.pivot_table(values=unsafe_col, index='scenario_id',
                       columns='pressure_id', aggfunc='mean')

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            vmin=0, vmax=1, linewidths=0.5)
ax.set_title('Unsafe Rate: Scenario x Pressure')
ax.set_ylabel('')
plt.tight_layout()
plt.savefig('../reports/figures/09_scenario_pressure_heatmap.png', bbox_inches='tight')
plt.show()

print('\n=== Key Findings ===')
print(f'1. Overall unsafe rate: {unsafe_rate:.1%} ({n_unsafe}/{len(df)} sessions)')
print(f'2. Most vulnerable scenario: {scenario_rates.index[0]} ({scenario_rates.iloc[0]:.1%})')
print(f'3. Highest-risk pressure: {pressure_rates.unsafe_rate.idxmax()} ({pressure_rates.unsafe_rate.max():.1%})')
print(f'4. Monitoring effect: monitored={mon_rates.loc["monitored","unsafe_rate"]:.1%} vs unmonitored={mon_rates.loc["unmonitored","unsafe_rate"]:.1%}')
print(f'5. Class balance for ML: {n_safe}:{n_unsafe} (safe:unsafe)')
print(f'6. Features available: {len(numeric_features)} numeric + 3 categorical')